# 1 - Bronze layer (Ingestion and saving to delta lake)

## 1.1 — List files in Unity Catalog Volume

In [0]:
import os

volume_path = "/Volumes/hr_catalogue_business_case/hr_raw/hr_data_businesscase/"

files = dbutils.fs.ls(volume_path)
for f in files:
    print(f.name)

In [0]:
base_path = "/Volumes/hr_catalogue_business_case/hr_raw/hr_data_businesscase"

files = {
    "absences":      "ABSENCES.csv",
    "absence_types": "Absence_Type.csv",
    "contracts":     "CONTRACT_BASIS.csv",
    "postcodes":     "POSTCODES.csv",
    "salary":        "SALARY_STATEMENT.csv",
    "workplan":      "WORK_PLAN.csv",
}

dfs = {}
for name, file in files.items():
    dfs[name] = (spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ";")
        .load(f"{base_path}/{file}"))
    print(f"{name}: {dfs[name].count()} rows")

In [0]:
dfs["contracts"].display()

## 1.6 Save in Delta Lake structure

In [0]:
# Rename columns with spaces before saving to Delta
from pyspark.sql import functions as F

def rename_cols(df):
    for col_name in df.columns:
        new_name = col_name.replace(" ", "_")
        df = df.withColumnRenamed(col_name, new_name)
    return df

In [0]:
for name, df in dfs.items():
    rename_cols(df).write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"hr_catalogue_business_case.hr_raw.{name}")

If anything fails in silver or gold, we use this table and not original one


In [0]:
%sql
DESCRIBE HISTORY hr_catalogue_business_case.hr_raw.contracts;   -- See versions after overwrite

In [0]:
%sql
SELECT count(*) AS TOTAL FROM hr_catalogue_business_case.hr_raw.contracts;